In [1]:
import os
import time
import pandas as pd
import io
import random
import numpy as np
from tqdm import tqdm
from together import Together
from dotenv import load_dotenv

def set_seed(seed: int):
    random.seed(seed) # Python
    np.random.seed(seed)  # Numpy, é o gerador utilizado pelo sklearn
    os.environ['PYTHONHASHSEED'] = str(seed)  # sistema operativo

set_seed(25)
load_dotenv()

True

In [2]:
# Initialize Together client
client = Together(api_key=os.getenv('TOGETHER_API_KEY'))
model = 'meta-llama/Llama-3.3-70B-Instruct-Turbo-Free'  # You can change this to any model available on Together
batch_size = 5
df = pd.read_csv('submission3_inputs.csv', sep=';')
output_path = 'submissao3-grupo001-s3.csv'

classification_prompt = (
    'You are an expert in distinguishing AI-generated text from Human written text.\n'
    'Analyze each provided text sample and classify it strictly as either "AI" or "Human".\n'
    'Note that I need the format to be a CSV with "ID" and "Label", and nothing else.\n'
    'Here are some classified examples:\n'
    "D2-100	It is an approximation useful in chemistry, but not strictly correct. It is not, in general, obeyed by nuclear reactions, or even in one common medical procedure, PET scanning (positron emission tomography). In this procedure, radioactive variants of common drug or body chemicals are injected into a patient under observation. Some of a particular radioactive chemical element or elements are radioactive in a particular way—they are neutron-light. Their nuclear reactions in the body are similar to beta-decay of neutron-heavy elements, except that the “electrons” released are not electrons, but the corresponding anti-matter: positrons. As anti-matter, positrons are quite unwelcome in ordinary matter—the patient—and very quickly encounter electrons, annihilating each electron-positron pair.,Human\n"
    "D2-11	These nutrients are needed to keep bones, teeth and muscles healthy. A lack of vitamin D can lead to bone deformities such as rickets in children, and bone pain caused by a condition called osteomalacia in adults. Government advice is that everyone should consider taking a daily vitamin D supplement during the autumn and winter. People at high risk of not getting enough vitamin D, all children aged 1 to 4, and all babies (unless they're having more than 500ml of infant formula a day) should take a daily supplement throughout the year. There have been some reports about vitamin D reducing the risk of coronavirus (COVID-19).,Human\n"
    "D2-12	Vitamin D is essential for maintaining healthy bones, teeth, and immune function, primarily by regulating calcium and phosphorus levels in the body. Humans can synthesize vitamin D through skin exposure to sunlight (UVB rays), but factors like limited sun exposure, darker skin, age, and geographic location can reduce production. The recommended daily intake varies by age and health status. For most adults, 600-800 IU (15-20 µg) per day is sufficient, though some individuals may require higher doses, especially if deficient. Dietary sources include fatty fish, fortified foods, and supplements.\nDeficiency can lead to rickets in children, osteomalacia in adults, and weakened immunity, making adequate vitamin D intake crucial for overall health and well-being. Regular checks can ensure optimal levels.,AI\n"
    "D2-13	Within 50 million years, the pressure and density of hydrogen in the center of the protostar became great enough for it to begin thermonuclear fusion.As helium accumulates at its core, the Sun is growing brighter; early in its main-sequence life its brightness was 70% that of what it is today.The temperature, reaction rate, pressure, and density increased until hydrostatic equilibrium was achieved: the thermal pressure counterbalancing the force of gravity. At this point, the Sun became a main-sequence star. Solar wind from the Sun created the heliosphere and swept away the remaining gas and dust from the protoplanetary disc into interstellar space.,Human\n"
    "D2-15	There are an estimated 2 trillion galaxies in the known Universe, some will hold millions and others will hold billions of stars. Most of the stars will have various amounts of planets in orbit around them and trillions of those planets will be in the Goldilocks Zone, which is an area in space that has the right conditions for liquid water to exist. So the logical thing you might say is “There must be life on some of them”. Maybe some other intelligent life form on a faraway world is asking that same question. If we are alone that would make us more special and unique than we could ever imagine.,Human\n"
    "D2-16	Type 2 diabetes rates have risen dramatically worldwide since the 1980s, constituting a global health crisis. The prevalence has more than doubled over the past three decades, increasing from approximately 108 million adults in 1980 to over 537 million (10.5% of global adult population) by 2021. This surge has been particularly pronounced in middle-income countries experiencing rapid urbanization, with Pacific Islands and Middle Eastern nations showing the highest prevalence rates, while China and India have the largest absolute numbers. Key drivers include rising obesity rates, increasingly sedentary lifestyles, Westernization of diets, aging populations, and improved detection methods. While some high-income countries have shown signs of stabilization recently, global projections remain concerning, with cases expected to reach 783 million by 2045.,AI\n"
    "D2-17	Quarks and gluons are both fundamental particles in the Standard Model of particle physics, but they have distinct roles. Quarks are the building blocks of matter, combining to form protons, neutrons, and other hadrons. They come in six flavors: up, down, charm, strange, top, and bottom. Quarks have fractional electric charges and interact via the strong nuclear force, which holds them together inside protons and neutrons. Gluons, on the other hand, are force carrier particles responsible for transmitting the strong nuclear force between quarks. Unlike quarks, gluons have no mass and no electric charge, but they do carry a property called color charge, which allows them to bind quarks together through a phenomenon known as color confinement.,AI\n"
    "D2-18	Insects as food or edible insects are insect species used for human consumption. Over 2 billion people are estimated to eat insects on a daily basis. Globally, more than 2,000 insect species are considered edible, though far fewer are discussed for industrialized mass production and regionally authorized for use in food. Many insects are highly nutritious, though nutritional content depends on species and other factors such as diet and age. Insects offer a wide variety of flavors and are commonly consumed whole or pulverized for use in dishes and processed food products such as burger patties, pasta, or snacks. Like other foods, there can be risks associated with consuming insects, such as allergic reactions.,Human\n"
    "D2-2	Spermidine is an aliphatic polyamine. Spermidine synthase (SPDS) catalyzes its formation from putrescine. It is a precursor to other polyamines, such as spermine and its structural isomer thermospermine. Spermidine synchronizes an array of biological processes, (such as Ca2+, Na+, K+ -ATPase) thus maintaining membrane potential and controlling intracellular pH and volume. Spermidine regulates biological processes, such as Ca2+ influx by glutamatergic N-methyl-D-aspartate receptor (NMDA receptor), which has been associated with nitric oxide synthase (NOS) and cGMP/PKG pathway activation and a decrease of Na+,K+-ATPase activity in cerebral cortex synaptosomes. Spermidine is a longevity agent in mammals due to various mechanisms of action, which are just beginning to be understood.,Human\n"
    "D2-21	The probability of extraterrestrial life existing remains one of science's great unknowns, with estimates varying widely based on different assumptions and models. While we lack direct evidence of life beyond Earth, several scientific factors inform our understanding: The Drake Equation, formulated in 1961, attempts to estimate the number of active, communicative extraterrestrial civilizations in our galaxy by multiplying several probability factors. However, many of these factors remain highly uncertain. Many astrobiologists consider microbial life likely to exist elsewhere, with estimates ranging from cautious to highly optimistic. The probability of intelligent civilizations remains more speculative, with the Fermi Paradox (questioning why we haven't detected them) suggesting either rarity or communication difficulties.,AI\n"
    "D2-22	Insects can be used for food production and are already consumed in many cultures worldwide. They are a nutritious, sustainable, and environmentally friendly protein source. Edible insects such as crickets, mealworms, grasshoppers, and black soldier fly larvae are rich in protein, healthy fats, vitamins, and minerals like iron and zinc. Compared to traditional livestock, insect farming requires less land, water, and feed, while producing lower greenhouse gas emissions. Insects can be used in whole form, ground into flours for protein bars, pasta, or baked goods, or processed into animal feed. While cultural barriers exist in some regions, the growing interest in sustainable food sources is driving research and commercial production of edible insects globally.,AI\n"
    "D2-23	Diabetes type 1 and type 2 are both chronic conditions affecting blood sugar regulation but have different causes and mechanisms. Type 1 diabetes is an autoimmune disease where the immune system attacks insulin-producing cells in the pancreas, leading to little or no insulin production. It usually develops in childhood or adolescence and requires lifelong insulin therapy. Type 2 diabetes is primarily caused by insulin resistance, where the body still produces insulin but cannot use it effectively. It is more common in adults and is strongly linked to obesity, diet, and lifestyle factors. Type 2 can often be managed with diet, exercise, and medication, while type 1 requires insulin replacement. Both types require careful blood sugar management to prevent complications.,AI\n"
    "D2-24	Entomophagy—the practice of consuming insects as food—has deep historical roots in many cultures worldwide, with approximately 2 billion people regularly incorporating insects into their diets. While Western societies have traditionally viewed insects with disgust, growing environmental and food security concerns have sparked renewed interest in this protein source.\nNutritionally, edible insects offer impressive benefits. Most species provide complete proteins containing all essential amino acids, often at higher concentrations than conventional meats. Many insects are rich in micronutrients like iron, zinc, and B vitamins, while containing healthy fats including omega-3 fatty acids. Cricket flour, for example, contains approximately 60-70% protein by weight, exceeding most plant and animal sources.,AI\n"
    "D2-25	Quarks and gluons are fundamental particles in the Standard Model of particle physics, both essential to our understanding of strong nuclear forces, yet they differ in several key ways. Fundamental Nature Quarks are elementary fermions that serve as the building blocks of hadrons (protons, neutrons, and other composite particles). Six flavors exist: up, down, charm, strange, top, and bottom. In contrast, gluons are gauge bosons that mediate the strong force between quarks, similar to how photons mediate the electromagnetic force. Properties Quarks possess fractional electric charges (+2/3 or -1/3), mass, spin-1/2, and color charge. Gluons are massless, have spin-1, carry no electric charge, but uniquely possess both color and anticolor charges.,AI\n"
    "D2-26	The global increase in demand for meat and the limited land area available prompt the search for alternative protein sources. Also the sustainability of meat production has been questioned. Edible insects as an alternative protein source for human food and animal feed are interesting in terms of low greenhouse gas emissions, high feed conversion efficiency, low land use, and their ability to transform low value organic side streams into high value protein products. More than 2000 insect species are eaten mainly in tropical regions. The role of edible insects in the livelihoods and nutrition of people in tropical countries is discussed, but this food source is threatened,Human\n"
    "D2-27	In 1802, Lamarck published Hydrogéologie, and became one of the first to use the term biology in its modern sense. In Hydrogéologie, Lamarck advocated a steady-state geology based on a strict uniformitarianism. He argued that global currents tended to flow from east to west, and continents eroded on their eastern borders, with the material carried across to be deposited on the western borders. Thus, the Earth's continents marched steadily westward around the globe. That year, he also published Recherches sur l'Organisation des Corps Vivants, in which he drew out his theory on evolution. He believed that all life was organized in a vertical chain, with gradation between the lowest forms and the highest forms of life.,Human\n"
    "D1-1	The cell cycle, or cell-division cycle, is the sequential series of events that take place in a cell that causes it to divide into two daughter cells. These events include the growth of the cell, duplication of its DNA (DNA replication) and some of its organelles, and subsequently the partitioning of its cytoplasm, chromosomes and other components into two daughter cells in a process called cell division. In eukaryotic cells (having a cell nucleus) including animal, plant, fungal, and protist cells, the cell cycle is divided into two main stages: interphase, and the M phase that includes mitosis and cytokinesis.,Human\n"
    "D1-2	The cell cycle is the process by which a cell grows, duplicates its DNA, and divides into two daughter cells. It is essential for growth, tissue repair, and reproduction. The cycle consists of four main phases. In G₁ phase, the cell grows, produces proteins, and prepares for DNA replication. During the S phase, DNA is replicated to ensure that each daughter cell receives an identical copy. The G2 phase follows, where the cell continues growing and checks for DNA damage before proceeding to division. Finally, in the M phase, the cell undergoes mitosis, where chromosomes are separated, and cytokinesis, where the cytoplasm splits.,AI\n"
    "D1-3	Photons, in many atomic models in physics, are particles which transmit light. In other words, light is carried over space by photons. Photon is an elementary particle that is its own antiparticle. In quantum mechanics each photon has a characteristic quantum of energy that depends on frequency: A photon associated with light at a higher frequency will have more energy (and be associated with light at a shorter wavelength).Photons have a rest mass of 0 (zero). However, Einstein's theory of relativity says that they do have a certain amount of momentum. Before the photon got its name, Einstein revived the proposal that light is separate pieces of energy (particles). These particles came to be known as photons.,Human\n"
    "D1-4	A photon is a fundamental particle of light and other electromagnetic radiation. It has no mass, no electric charge, and always moves at the speed of light (299,792,458 m/s in a vacuum). Photons carry energy and momentum, which depend on their wavelength or frequency. Higher frequency photons (like X-rays) have more energy, while lower frequency photons (like radio waves) have less. Their energy is given by Planck’s equation:E = h f, where E is energy, h is Planck’s constant, and f is frequency.Photons behave as both particles and waves (wave-particle duality), meaning they can interfere, diffract, and be absorbed/emitted like particles. They are responsible for vision, photosynthesis, solar power, and many quantum phenomena.,AI\n"
    "D1-5	According to the theory of plate tectonics, Earth's lithosphere, its rigid outer shell, is broken into sixteen larger and several smaller plates. These move continuously at a slow pace, due to convection in the underlying ductile mantle, and most volcanic activity on Earth takes place along plate boundaries, where plates are converging (and lithosphere is being destroyed) or are diverging (and new lithosphere is being created). During the development of geological theory, certain concepts that allowed the grouping of volcanoes in time, place, structure and composition have developed that ultimately have had to be explained in the theory of plate tectonics.,Human\n"
    "D1-6	The theory of plate tectonics explains that Earth’s lithosphere is divided into moving plates that float on the semi-fluid asthenosphere. These movements shape the planet, causing earthquakes, volcanic activity, and mountain formation. The theory builds on continental drift, proposed by Alfred Wegener, and is supported by evidence from seafloor spreading, fossil distribution, and geological formations. Plates move apart at divergent boundaries, collide at convergent boundaries, and slide past each other at transform boundaries. Their movement is driven by mantle convection, gravity, and Earth’s rotation, constantly reshaping the surface. This theory is essential for understanding natural disasters and the geological evolution of continents.,AI\n"
    "D1-7	Thalidomide is a pharmaceutical drug, first prepared in 1957 in Germany, prescribed for treating morning sickness in pregnant women. The drug was discovered to be teratogenic, causing serious genetic damage to early embryonic growth and development, leading to limb deformation in babies. Several proposed mechanisms of teratogenicity involve different biological functions for the (R)- and (S)-thalidomide enantiomers.In the human body, however, thalidomide undergoes racemization: even if only one of the two enantiomers is administered as a drug, the other enantiomer is produced as a result of metabolism. Thalidomide is currently used for the treatment of other diseases, notably cancer and leprosy. Strict regulations and controls have been implemented to avoid its use by pregnant women and prevent developmental deformities.,Human\n"
    "D1-8   Thalidomide is a drug that was first developed in the 1950s as a sedative and later prescribed to pregnant women for morning sickness. However, it caused severe birth defects, including missing or malformed limbs, when taken during pregnancy. The tragedy affected thousands of babies and led to stricter drug regulations worldwide. Despite its harmful past, thalidomide was later found to have anti-inflammatory and immunomodulatory properties. Today, it is used under strict controls to treat multiple myeloma, leprosy-related inflammation, and some autoimmune diseases. Due to its risks, it is only prescribed under a controlled program to prevent use during pregnancy. Its history remains a major lesson in pharmaceutical safety.,AI\n"
    "D1-9	Kepler published his first two laws about planetary motion in 1609, having found them by analyzing the astronomical observations of Tycho Brahe.Kepler's third law was published in 1619. Kepler had believed in the Copernican model of the Solar System, which called for circular orbits, but he could not reconcile Brahe's highly precise observations with a circular fit to Mars' orbit – Mars coincidentally having the highest eccentricity of all planets except Mercury. His first law reflected this discovery. In 1621, Kepler noted that his third law applies to the four brightest moons of Jupiter. Godefroy Wendelin also made this observation in 1643.,Human\n"
    "D1-10	Kepler’s laws of planetary motion are three fundamental principles that describe the motion of planets around the Sun. These laws were formulated by the German astronomer Johannes Kepler in the early 17th century, building on the detailed observations of the astronomer Tycho Brahe. Kepler’s work came at a time when the heliocentric model (Sun-centered solar system) proposed by Copernicus was still controversial. Kepler initially struggled with the Copernican model, but after inheriting Brahe’s precise astronomical data, he was able to make groundbreaking discoveries. Kepler’s first law, the Law of Ellipses, stated that planets move in elliptical orbits with the Sun at one focus. The second, the Law of Equal Areas, explained that planets sweep out equal areas in equal times.,AI\n"
    "D1-11	The basic idea of biological evolution is that populations and species of organisms change over time. Today, when we think of evolution, we are likely to link this idea with one specific person: the British naturalist Charles Darwin. In the 1850s, Darwin wrote an influential and controversial book called On the Origin of Species. In it, he proposed that species evolve (or, as he put it, undergo descent with modification), and that all living things can trace their descent to a common ancestor. Darwin also suggested a mechanism for evolution: natural selection, in which heritable traits that help organisms survive and reproduce become more common in a population over time.,Human\n"
    "D1-12	Biological evolution, according to Charles Darwin, refers to the process by which species of organisms change over time through variations in traits that are passed down from one generation to the next. Darwin’s theory of evolution by natural selection suggests that individuals within a species show variation in their characteristics. These variations can affect their ability to survive and reproduce in their environment. Those individuals with traits that give them a survival or reproductive advantage are more likely to pass those traits on to their offspring, while less advantageous traits are gradually eliminated. Over long periods, this process leads to the accumulation of beneficial traits in the population, resulting in the adaptation of species to their environment.,AI\n"
    "D1-13	Stoichiometry rests upon the very basic laws that help to understand it better, i.e., law of conservation of mass, the law of definite proportions (i.e., the law of constant composition), the law of multiple proportions and the law of reciprocal proportions. In general, chemical reactions combine in definite ratios of chemicals. Since chemical reactions can neither create nor destroy matter, nor transmute one element into another, the amount of each element must be the same throughout the overall reaction. For example, the number of atoms of a given element X on the reactant side must equal the number of atoms of that element on the product side, whether or not all of those atoms are actually involved in a reaction.,Human\n"
    "D1-14	Stoichiometry is the branch of chemistry that deals with the calculation of reactants and products in chemical reactions. It is based on the concept that matter is conserved during a reaction, meaning the quantity of reactants used equals the quantity of products formed. Stoichiometry involves using balanced chemical equations to determine the molar ratios between substances, allowing the calculation of how much of each reactant is needed and how much product will be formed. This includes conversions between moles, mass, and volume. Stoichiometric calculations are essential for understanding reaction yields, determining the limiting reactant, and ensuring efficient use of materials in chemical processes.,AI\n"
    "D1-15	General relativity is a theory of gravitation developed by Einstein in the years 1907–1915. The development of general relativity began with the equivalence principle, under which the states of accelerated motion and being at rest in a gravitational field (for example, when standing on the surface of the Earth) are physically identical. The upshot of this is that free fall is inertial motion: an object in free fall is falling because that is how objects move when there is no force being exerted on them, instead of this being due to the force of gravity as is the case in classical mechanics.,Human\n"
    "D1-16	General relativity, also known as the general theory of relativity, is a geometric theory of gravitation developed by Albert Einstein between 1907 and 1915. It provides a unified description of gravity as a property of spacetime, which is a four-dimensional continuum combining space and time. The theory posits that massive objects cause a curvature in spacetime, and this curvature influences the motion of objects, effectively manifesting as gravity. Einstein developed general relativity to address limitations in Newton's law of universal gravitation, particularly in explaining anomalies like the precession of Mercury's orbit. The theory has been extensively tested and confirmed through observations of gravitational waves, the bending of light around massive objects, and the behavior of objects in strong gravitational fields.,AI\n"
    "D1-17	The recommended number of eggs to consume per week varies depending on several factors, including individual health, dietary preferences, and overall dietary patterns. It's important to note that while eggs are a nutritious food, they are also relatively high in dietary cholesterol. According to general guidelines, including those provided by the American Heart Association and the Dietary Guidelines for Americans, consuming up to seven eggs per week is considered reasonable and can be part of a healthy diet for most people. However, it's important to consider the overall context of your diet, including other sources of cholesterol and saturated fat, as well as your individual health goals and any specific dietary restrictions or considerations you may have.,Human\n"
    "D1-18	The question of how many eggs one should eat per day is often debated due to concerns about cholesterol and cardiovascular health. Current research suggests that consuming up to three eggs per day can be part of a healthy diet for most people, particularly young, healthy adults. Consuming up to three eggs per day has been shown to increase HDL (good) cholesterol and improve the LDL/HDL ratio, which are favorable changes for cardiovascular health. For most healthy individuals, consuming up to three eggs per day can be beneficial, improving cholesterol profiles and providing essential nutrients without increasing cardiovascular risk. However, individuals with specific health conditions, such as cardiovascular disease or diabetes, should consider their overall dietary patterns.,AI\n"
    "D1-19	There seems to be some merit to having a single daily glass of red wine, it seems to have something called resveratrol, which acts as an antioxidant and is thought to be good for your heart. Additionally, Italian researchers found that moderate beer drinkers had a 42 percent lower risk of heart disease compared to non-drinkers. For maximum protection, keep your consumption to one pint—at around 5 percent alcohol by volume—a day, the researchers say. Basically, alcohol, in moderation, appears to lower risks of heart and other cardiovascular disease.But this is in moderation, mind you. Just as with everything else, the only difference between medicine and poison is the dosage. A beer a day is good; twelve is not.,Human\n"
    "D1-20	No level of alcohol consumption is considered entirely safe for health. According to the World Health Organization (WHO), even low levels of alcohol use can increase the risk of certain cancers and other health problems. While some guidelines suggest limits to reduce risks, these guidelines are based on minimizing risk rather than ensuring safety. Recent research indicates that even moderate drinking may increase the risk of chronic diseases and premature death compared to abstaining. Therefore, the safest approach is to avoid alcohol altogether. Alcohol consumption significantly impacts mental health, often exacerbating existing conditions and contributing to new ones. Alcohol initially acts as a depressant, reducing inhibitions and creating a temporary sense of relaxation or confidence.,AI\n"
    "D1-21	There are three major compounds of life: proteins; lipids and carbohydrates. The perception of sweet taste, mainly associated with advantageous food, has had an important evolutionary influence on different physiological regulation mechanisms. During human development, sugar was always luxury. In 1885 Constantin Fahlberg produced the first artificial sweetener, saccharin, and the scientific establishment was surprised by its extreme sweetness. Significant to this discovery was the fact that sweet taste became affordable to poor people. Following the commercial success of artificial sweeteners, a battle between the sugar and sweetener industries began. Saccharin was claimed to be carcinogenic in rats. However, it was later shown that saccharin is neither toxic nor carcinogenic in normal amounts, yet its reputation remains tarnished.,Human\n"
    "D1-22	Artificial sweeteners are low- or zero-calorie sugar substitutes used to sweeten foods and beverages without the added calories of sugar. Common examples include aspartame, sucralose, saccharin, and stevia (a natural alternative). They are widely used in diet sodas, sugar-free gum, and diabetic-friendly products. While they provide sweetness without spiking blood sugar levels, they remain controversial. Studies show they are safe in moderation, but concerns exist about potential long-term effects on gut health and metabolism. Some people experience sensitivity to specific sweeteners. Ultimately, artificial sweeteners can be helpful for reducing calorie intake, but their impact varies widely from person to person.,AI\n"
    "D1-23	Recent findings on the ecology, etiology and pathology of coral pathogens, host resistance mechanisms, previously unknown disease/syndromes and the global nature of coral reef diseases have increased our concern about the health and future of coral reef communities. Much of what has been discovered in the past 4 years is presented in this special issue. Among the significant findings, the role that various Vibrio species play in coral disease and health, the composition of the ‘normal microbiota’ of corals, and the possible role of viruses in the disease process are important additions to our knowledge.  reefs and a major cause of reef deterioration.,Human\n"
)

responses = []
total_samples = len(df)
total_batches = (total_samples + batch_size - 1) // batch_size

print(f'Processing {total_samples} samples in {total_batches} batches...')

for batch_idx in range(total_batches):
    start, end = batch_idx * batch_size, min((batch_idx + 1) * batch_size, total_samples)

    batch_texts = '\n\n'.join([f'Text {row.ID}: {row.Text}' for row in df.iloc[start:end].itertuples(index=False)])

    user_query = f'Classify the following texts as Human or AI:\n\n{batch_texts}\n\nReturn the result as a CSV with "ID" and "Label". Ensure the IDs match exactly. Make sure It is just the ID and Label, dont add anything else'

    max_retries, delay = 10, 5
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {'role': 'system', 'content': classification_prompt},
                    {'role': 'user', 'content': user_query}
                ],
                max_tokens=200,
                temperature=0.0 
            )

            csv_output = response.choices[0].message.content.strip()

            print(f'Batch {batch_idx + 1} API Response:\n{csv_output}\n')
            responses.append(f'Batch {batch_idx + 1}:\n{csv_output}\n\n')
            break
        except Exception as e:
            print(f'Attempt {attempt + 1} failed: {e}')
            if attempt < max_retries - 1:
                print(f'Retrying in {delay} seconds...')
                time.sleep(delay)
                delay *= 2
            else:
                responses.append(f'Batch {batch_idx + 1}: Error after max retries.\n\n')
                print('Max retries reached. Moving to next batch.')
    
    time.sleep(65)

with open(output_path, 'w', encoding='utf-8') as file:
    file.writelines(responses)

print(f'Classification completed. Raw responses saved to {output_path}')

Processing 100 samples in 20 batches...
Batch 1 API Response:
ID,Label
D3-1,AI
D3-2,AI
D3-3,AI
D3-4,Human
D3-5,Human

Batch 2 API Response:
D3-6,AI
D3-7,Human
D3-8,AI
D3-9,AI
D3-10,AI

Batch 3 API Response:
ID,Label
D3-11,AI
D3-12,AI
D3-13,AI
D3-14,AI
D3-15,AI

Batch 4 API Response:
ID,Label
D3-16,Human
D3-17,AI
D3-18,Human
D3-19,Human
D3-20,Human

Batch 5 API Response:
ID,Label
D3-21,AI
D3-22,Human
D3-23,AI
D3-24,AI
D3-25,AI

Batch 6 API Response:
ID,Label
D3-26,Human
D3-27,Human
D3-28,AI
D3-29,AI
D3-30,AI

Batch 7 API Response:
ID,Label
D3-31,AI
D3-32,AI
D3-33,Human
D3-34,AI
D3-35,AI

Batch 8 API Response:
ID,Label
D3-36,AI
D3-37,AI
D3-38,AI
D3-39,AI
D3-40,AI

Batch 9 API Response:
ID,Label
D3-41,AI
D3-42,Human
D3-43,AI
D3-44,Human
D3-45,AI

Batch 10 API Response:
D3-46,AI
D3-47,Human
D3-48,AI
D3-49,Human
D3-50,AI

Batch 11 API Response:
ID,Label
D3-51,Human
D3-52,Human
D3-53,AI
D3-54,AI
D3-55,AI

Batch 12 API Response:
ID,Label
D3-56,AI
D3-57,AI
D3-58,Human
D3-59,AI
D3-60,AI

Batch 